In [9]:
import pandas as pd

phrasebank_df = pd.read_csv(
    'FinancialPhraseBank-v1.0/Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['text', 'sentiment'],
    encoding='latin-1'  # needed for special characters
)

phrasebank_df

,text,sentiment
0,"According to Gran , the company has no plans t...",neutral
1,With the new production plant the company woul...,positive
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,"In the third quarter of 2010 , net sales incre...",positive
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive
...,...,...
3448,Operating result for the 12-month period decre...,negative
3449,HELSINKI Thomson Financial - Shares in Cargote...,negative
3450,LONDON MarketWatch -- Share prices ended lower...,negative
3451,Operating profit fell to EUR 35.4 mn from EUR ...,negative


In [8]:
phrasebank_df['sentiment'].value_counts()

sentiment
neutral     2146
positive     887
negative     420
Name: count, dtype: int64

In [9]:
label_map = {"neutral": 2, "positive": 1, "negative": 0}
phrasebank_df['sentiment'] = phrasebank_df['sentiment'].map(label_map)

phrasebank_df.rename(columns={"sentiment": "label"}, inplace=True)
phrasebank_df

,text,label
0,"According to Gran , the company has no plans t...",2
1,With the new production plant the company woul...,1
2,"For the last quarter of 2010 , Componenta 's n...",1
3,"In the third quarter of 2010 , net sales incre...",1
4,Operating profit rose to EUR 13.1 mn from EUR ...,1
...,...,...
3448,Operating result for the 12-month period decre...,0
3449,HELSINKI Thomson Financial - Shares in Cargote...,0
3450,LONDON MarketWatch -- Share prices ended lower...,0
3451,Operating profit fell to EUR 35.4 mn from EUR ...,0


Tweets dataset already to integers:
0 -> bearish (negative)
1 -> bullish (positive)
2 -> neutral 

In [10]:
phrasebank_df['label'].value_counts()

label
2    2146
1     887
0     420
Name: count, dtype: int64

In [11]:
full_twitter['label'].value_counts()

NameError: name 'full_twitter' is not defined

In [36]:
df_merged = pd.concat([phrasebank_df, full_twitter], ignore_index=True, sort=False)
df_merged

,text,label
0,"According to Gran , the company has no plans t...",2
1,With the new production plant the company woul...,1
2,"For the last quarter of 2010 , Componenta 's n...",1
3,"In the third quarter of 2010 , net sales incre...",1
4,Operating profit rose to EUR 13.1 mn from EUR ...,1
...,...,...
15379,Stocks making the biggest moves midday: TD Ame...,2
15380,Stocks making the biggest moves premarket: Fit...,2
15381,Stocks making the biggest moves premarket: Hom...,2
15382,Stocks making the biggest moves premarket: TD ...,2


In [37]:
df_merged['label'].value_counts()

label
2    9890
1    3285
0    2209
Name: count, dtype: int64

In [10]:
phrasebank_df = pd.read_csv(
    'FinancialPhraseBank-v1.0/Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['text', 'sentiment'],
    encoding='latin-1'  # needed for special characters
)

label_map = {"neutral": 2, "positive": 1, "negative": 0}
phrasebank_df['sentiment'] = phrasebank_df['sentiment'].map(label_map)

phrasebank_df.rename(columns={"sentiment": "label"}, inplace=True)

splits = {'train': 'sent_train.csv', 'validation': 'sent_valid.csv'}
df_tr = pd.read_csv("hf://datasets/zeroshot/twitter-financial-news-sentiment/" + splits["train"])
df_val = pd.read_csv("hf://datasets/zeroshot/twitter-financial-news-sentiment/" + splits["validation"])
twitter_df = pd.concat([df_tr, df_val], ignore_index=True)

total_df = pd.concat([phrasebank_df, twitter_df], ignore_index=True, sort=False)

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'\$[a-zA-Z]+', 'TICKER', text) # normalize tickers
    return text

import pandas as pd

pd.set_option('display.max_colwidth', None)

total_df["text"] = total_df["text"].apply(clean_text)
total_df[2059:2097]


,text,label
2059,negotiations with representatives of the personnel regarding the restructuring process have now been ended .,2
2060,neither of the companies use genetically engineered soy at the moment .,2
2061,"neste oil will publish its third quarter 2008 results on friday , 24 october 2008 at approximately 9 am ( eet ) .",2
2062,new novator products are supposed to be exported .,2
2063,"niina nenonen , marimekko 's current director for clothing , bags and accessories lines , will take up this role .",2
2064,no changes in media activity were seen in october compared with september .,2
2065,no decision on such sale of the now issued or existing treasury shares to ya global has been made yet .,2
2066,no financial detail were available .,2
2067,no financial details were revealed .,2
2068,no financial information was provided .,2


In [11]:
# Class distribution
print(total_df["label"].value_counts(normalize=True))

# Word count stats per bucket
total_df["word_count"] = total_df["text"].str.split().str.len()
print(total_df["word_count"].describe())

# Length bucket distribution
def assign_bucket(wc):
    if wc <= 8: return "short"
    elif wc <= 20: return "medium"
    else: return "long"

total_df["bucket"] = total_df["word_count"].apply(assign_bucket)
print(total_df["bucket"].value_counts())

label
2    0.642876
1    0.213534
0    0.143591
Name: proportion, dtype: float64
count    15384.000000
mean        14.158736
std          7.700136
min          0.000000
25%          9.000000
50%         12.000000
75%         17.000000
max         81.000000
Name: word_count, dtype: float64
bucket
medium    10148
short      3028
long       2208
Name: count, dtype: int64


In [12]:
# See the distribution
print(total_df["word_count"].quantile([0.90, 0.95, 0.99]))

0.90    23.0
0.95    30.0
0.99    43.0
Name: word_count, dtype: float64


In [1]:
from dataset import create_dataframe, split_dataset, clean_text
from svm import SVM_Classifier

In [2]:
#Step 1: create dataframe based on the image paths
df = create_dataframe()
df['text'] = df['text'].apply(clean_text)
print(df)

                                                    text  label
0      according to gran , the company has no plans t...      2
1      with the new production plant the company woul...      1
2      for the last quarter of 2010 , componenta 's n...      1
3      in the third quarter of 2010 , net sales incre...      1
4      operating profit rose to eur 13.1 mn from eur ...      1
...                                                  ...    ...
15379  stocks making the biggest moves midday: td ame...      2
15380  stocks making the biggest moves premarket: fit...      2
15381  stocks making the biggest moves premarket: hom...      2
15382  stocks making the biggest moves premarket: td ...      2
15383         tco, nnvc, gpor and je among midday movers      2

[15384 rows x 2 columns]


In [3]:
#Step 2: Split the dataset
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset(df)


In [4]:
#Step 3: SVM with TF-IDF
svmclass = SVM_Classifier()
svmclass.fit(X_train, y_train)
y_pred = svmclass.predict(X_test)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'C': 1, 'kernel': 'rbf'}
Best macro F1: 0.7485001971647353


In [5]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Overall metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 3))

# Per class breakdown
print(classification_report(y_test, y_pred, target_names=["negative", "positive", "neutral"]))

Accuracy: 0.834
Macro F1: 0.766
              precision    recall  f1-score   support

    negative       0.76      0.58      0.66       221
    positive       0.75      0.74      0.74       329
     neutral       0.87      0.92      0.90       989

    accuracy                           0.83      1539
   macro avg       0.79      0.75      0.77      1539
weighted avg       0.83      0.83      0.83      1539



In [6]:
from evaluate import (
    evaluate_overall,
    evaluate_by_bucket, 
    plot_confusion_matrix,
    build_results_table,
    plot_bucket_comparison
)

# After each model
overall = evaluate_overall(y_test, y_pred, model_name="SVM")
buckets = evaluate_by_bucket(y_test, y_pred, df["bucket"], model_name="SVM")
plot_confusion_matrix(y_test, y_pred, model_name="SVM")


SVM — Overall Results
Accuracy:  0.834
Macro F1:  0.766

              precision    recall  f1-score   support

    negative       0.76      0.58      0.66       221
    positive       0.75      0.74      0.74       329
     neutral       0.87      0.92      0.90       989

    accuracy                           0.83      1539
   macro avg       0.79      0.75      0.77      1539
weighted avg       0.83      0.83      0.83      1539



KeyError: 'bucket'